In [1]:
import os
import sys
sys.path.append('..')
os.environ['CUDA_VISIBLE_DEVICES'] = "1"


### Load a checkpoint for the task: QA1 | QA2 | QA3


In [ ]:
# os.makedirs('models', exist_ok=True)

In [ ]:
# !wget https://huggingface.co/booydar/rmt-gpt2-babilong-qa1/resolve/main/pytorch_model.bin -P models/
# !wget https://huggingface.co/booydar/rmt-gpt2-babilong-qa2/resolve/main/pytorch_model.bin -P models/
# !wget https://huggingface.co/booydar/rmt-gpt2-babilong-qa3/resolve/main/pytorch_model.bin -P models/

--2024-08-09 12:00:17--  https://huggingface.co/booydar/rmt-gpt2-babilong-qa1/resolve/main/pytorch_model.bin
Resolving huggingface.co (huggingface.co)... 18.239.50.103, 18.239.50.16, 18.239.50.49, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.103|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.huggingface.co/repos/2b/a6/2ba65b402b075ff6c41f583ffeea745cccc04e1aa48ec0a7895dd3ec95947c08/a2de2b7590e193a6c6d48cfe5b8b684643307fbe0655726986e456f7da17370c?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27pytorch_model.bin%3B+filename%3D%22pytorch_model.bin%22%3B&response-content-type=application%2Foctet-stream&Expires=1723464017&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcyMzQ2NDAxN319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmh1Z2dpbmdmYWNlLmNvL3JlcG9zLzJiL2E2LzJiYTY1YjQwMmIwNzVmZjZjNDFmNTgzZmZlZWE3NDVjY2NjMDRlMWFhNDhlYzBhNzg5NWRkM2VjOTU5NDdjMDgvYTJkZTJiNzU5MGUxOTNhNm

In [ ]:
#!git clone https://github.com/booydar/recurrent-memory-transformer/ -b babilong-release github
#sys.path.append('./github')

Cloning into 'github'...
remote: Enumerating objects: 5549, done.
remote: Counting objects: 100% (1826/1826), done.
remote: Compressing objects: 100% (542/542), done.
remote: Total 5549 (delta 1345), reused 1686 (delta 1277), pack-reused 3723
Receiving objects: 100% (5549/5549), 79.74 MiB | 6.76 MiB/s, done.
Resolving deltas: 100% (3775/3775), done.


### Create RMT model

In [2]:
#!pip install datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from modeling_rmt.language_modeling import RecurrentWrapper, MemoryCell
from datasets import load_dataset

/home/jovyan/rmt_it/conda_rmt_it_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = 'cpu'
tokenizer = AutoTokenizer.from_pretrained('gpt2')

model = AutoModelForCausalLM.from_pretrained('gpt2')
cell = MemoryCell(model, num_mem_tokens=5)#16)

model = RecurrentWrapper(cell,
                         segment_size=1024-10,
                         max_n_segments=16,
).to(device)


model_cpt = "../test_exps/rmt_gpt_2_ift_LA/checkpoint-5000/pytorch_model.bin"
cpt = torch.load(model_cpt, map_location=device)
model.load_state_dict(cpt, strict=False)

/home/jovyan/rmt_it/conda_rmt_it_venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/tmp/ipykernel_353623/4256426403.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed 

<All keys matched successfully>

### Prepare dataset

In [5]:
task = 'qa1'
seq_len = '0k'
ds = load_dataset('RMT-team/babilong', seq_len)[task]

Generating qa1 split: 100 examples [00:00, 174.51 examples/s]
Generating qa2 split: 100 examples [00:00, 6215.72 examples/s]
Generating qa3 split: 100 examples [00:00, 4831.70 examples/s]
Generating qa4 split: 100 examples [00:00, 1070.70 examples/s]
Generating qa5 split: 100 examples [00:00, 5612.61 examples/s]
Generating qa6 split: 100 examples [00:00, 999.07 examples/s]
Generating qa7 split: 100 examples [00:00, 1347.82 examples/s]
Generating qa8 split: 100 examples [00:00, 9163.47 examples/s]
Generating qa9 split: 100 examples [00:00, 4226.47 examples/s]
Generating qa10 split: 100 examples [00:00, 5973.86 examples/s]
Generating qa11 split: 100 examples [00:00, 1154.49 examples/s]
Generating qa12 split: 100 examples [00:00, 789.49 examples/s]
Generating qa13 split: 100 examples [00:00, 3095.29 examples/s]
Generating qa14 split: 100 examples [00:00, 5223.36 examples/s]
Generating qa15 split: 100 examples [00:00, 1032.64 examples/s]
Generating qa16 split: 100 examples [00:00, 792.41 e

In [42]:
sample_num = 9
sample = ds[sample_num]

In [43]:
sample

{'input': 'John journeyed to the hallway. John journeyed to the bedroom.',
 'question': 'Where is John? ',
 'target': 'bedroom'}

### Get prediction

In [44]:
input_text = f"{sample['input']} {sample['question']}"
model_input = tokenizer.encode_plus(input_text, return_tensors='pt').to(device)
out = model.generate(**model_input, max_new_tokens=20)
prediction = tokenizer.decode(out[0][:-1], add_special_tokens=False)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [45]:

print(f"Task: {input_text[:100]}\nPrediction: {prediction}\nAnswer: {sample['target']}\n{'Correct!' if sample['target'] == prediction else 'Wrong'}")

Task: John journeyed to the hallway. John journeyed to the bedroom. Where is John? 
Prediction:  the doorbell rang.  John was in the bedroom.  John was in the bedroom.
Answer: bedroom
Wrong
